### A. Using the mathematical representations developed in Task 2, write a program in either Python or R to solve the optimization problem computationally.

The goal is to determine how many tons Amazon should ship on each available route so that all fulfillment center demand is met at the lowest possible total cost.

In [ ]:
import pandas as pd
from pulp import (LpProblem, LpMinimize, LpVariable, lpSum, LpStatus, value)


# reading data from the excel
file_path = "Task3.xlsx"

centers = pd.read_excel(file_path, sheet_name="Centers").set_index("center_id")
hubs = pd.read_excel(file_path, sheet_name="Hubs").set_index("hub_id")
focus = pd.read_excel(file_path, sheet_name="Focus Cities").set_index("focus_id")
cost = pd.read_excel(file_path, sheet_name="Cost", na_values=["N/A"]).set_index("destination_id")

# creating dictionaries for costs
hub_focus_cost = {
    (h, f): cost.at[f, h]
    for h in hubs.index
    for f in focus.index
    if pd.notna(cost.at[f, h])
}

hub_center_cost = {
    (h, c): cost.at[c, h]
    for h in hubs.index
    for c in centers.index
    if pd.notna(cost.at[c, h])
}

focus_center_cost = {
    (f, c): cost.at[c, f]
    for f in focus.index
    for c in centers.index
    if pd.notna(cost.at[c, f])
}

# Creating the model
model = LpProblem("Amazon_Transportation", LpMinimize)

x = LpVariable.dicts("hub_to_focus", hub_focus_cost, lowBound=0)
y = LpVariable.dicts("hub_to_center", hub_center_cost, lowBound=0)
z = LpVariable.dicts("focus_to_center", focus_center_cost, lowBound=0)
